# 05 — Dependencies: what the agent can see

**Scope**

So far the model chose everything a tool used. Real tools also need facts the model must *not* choose:
who the user is, their account, an HTTP client. Pydantic AI calls these **dependencies** ("deps"). You
hand them to each run, and only your code can read them.

When you finish you can:

- give an agent a deps *type*, and hand each run its own deps
- read deps inside a tool with `RunContext` and `ctx.deps`
- write instructions that change per run with `@agent.instructions`
- say what gets stored in the message history, and why that matters

In [ ]:
from learningdantic import setup

MODEL = setup()

## Deps: data each run brings with it

Three pieces:

- `deps_type=BankDeps` on the `Agent`: the *kind* of deps every run must bring.
- `deps=BankDeps(...)` on `agent.run(...)`: the deps for *this* run.
- `ctx.deps` inside a tool: how the tool reads them.

New in the next cell:

- `@dataclass` turns a class with typed fields into a simple data holder, so `BankDeps("Anne", 1250.50)` just works.
- `@agent.tool` (without `_plain`) hands your tool a first argument, `ctx`: the run's context.
  `ctx: RunContext[BankDeps]` is a type hint that says "this context's `.deps` is a `BankDeps`".
- The model never sees `ctx`. It can ask for *the* balance, but it can't choose *whose*. That's scope.

In [ ]:
from dataclasses import dataclass

from pydantic_ai import Agent, RunContext


@dataclass
class BankDeps:
    customer_name: str
    balance: float
    currency: str = "USD"


bank_agent = Agent(
    MODEL,
    deps_type=BankDeps,
    instructions="You are helpful bank support. Look up balances with your tool; never guess them.",
)


@bank_agent.tool
def get_balance(ctx: RunContext[BankDeps]) -> str:
    """Get the current customer's balance."""
    return f"{ctx.deps.balance:.2f} {ctx.deps.currency}"


for name, balance in [("Anne", 1250.50), ("Bob", 42.00)]:
    result = await bank_agent.run(
        "How much money do I have right now?", deps=BankDeps(name, balance)
    )
    print(f"{name} → {result.output}")

> ✅ **Checkpoint** — two lines, such as `Anne → You have **$1,250.50 USD** right now.` and
> `Bob → You have **$42.00** right now.` (the `**` is the model's Markdown for bold; wording varies).
> Same agent, same tool, same question: only the deps changed.

## Instructions that change per run

A string in `instructions=` is fixed when you build the agent. A function decorated with
`@agent.instructions` runs at the start of every run, gets the same `ctx`, and its text is added to the
instructions.

This cell rebuilds the agent and adds the function in the same cell. Run it twice and you still get one
agent with the instruction once, not twice. `tools=[get_balance]` hands the tool from the cell above to
the new agent: a tool is just a function.

In [ ]:
bank_agent = Agent(
    MODEL,
    deps_type=BankDeps,
    instructions="You are helpful bank support. Look up balances with your tool; never guess them.",
    tools=[get_balance],
)


@bank_agent.instructions
def greet_by_name(ctx: RunContext[BankDeps]) -> str:
    return f"The customer's name is {ctx.deps.customer_name}. Address them by name."


r = await bank_agent.run("Do I have enough for a $60 game?", deps=BankDeps("Anne", 1250.50))
print(r.output)

> ✅ **Checkpoint** — the reply uses Anne's name and her balance, such as `Yes, Anne—you have $1,250.50, so
> after a $60 game you’d have $1,190.50 left.` You never typed "Anne" into the prompt.

## What gets stored in the history

`r.all_messages()` is the run's full message history (notebook 07 builds conversations from it). The deps
object itself is never stored there: every run reads its own deps. But what deps *produce* is stored: the
instructions as they were rendered, and every tool result.

In [ ]:
history = r.all_messages()
print("Instructions the model saw:")
print(history[0].instructions)
print()
print("Anne's balance is stored in the history:", "1250.50" in str(history))

> ✅ **Checkpoint** — the instructions include `The customer's name is Anne. Address them by name.`, and the
> last line ends in `True`.
>
> So if you save a history (notebook 07 does), treat it as user data: it can hold names and balances.

## Why bother?

- **Testing.** Notebook 08 runs agents against a fake model. `agent.override(deps=...)` swaps in fake deps
  the same way, so a test never touches real accounts or the network.
- **Many users, one agent.** Build the agent once and give each user's run that user's deps.
  Nothing leaks from one run to the next.

## 🏋️ Exercise — deps that carry an HTTP client

Use the free [Open-Meteo API](https://open-meteo.com) (no key needed for non-commercial use):

- geocode a city: `https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1`
- current temperature: `https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&current=temperature_2m`

1. Write `@dataclass class WeatherDeps: client: httpx.Client` and a `weather_agent` with two tools,
   `geocode` and `current_temp`, that call those URLs through `ctx.deps.client`. The tools stay plain `def`:
   `httpx.Client` is not async. In each tool, call `response.raise_for_status()` and turn any
   `httpx.HTTPError` into a `ModelRetry`, so a failed request becomes a retry instead of a crash.
2. Open the client with `with httpx.Client(timeout=10) as client:` (it closes itself when the block ends),
   and ask **"Is it warmer right now: Paris or Tokyo?"**. Both temperatures must come from your tools.

Stretch: add `unit: str = "celsius"` to `WeatherDeps` and pass it to Open-Meteo as `temperature_unit`.
Now `WeatherDeps(client, unit="fahrenheit")` changes the answer without touching the agent.

Your take: [`../solutions/05_httpx_deps.ipynb`](../solutions/05_httpx_deps.ipynb).

## Next →

→ **[06_logfire_observability.ipynb](06_logfire_observability.ipynb)** — watch your agents run, step by step,
in Logfire. It's optional; skip to **[07_history_and_streaming.ipynb](07_history_and_streaming.ipynb)** if you like.